In [1]:
import torch
from sorl.gat_sim import GAT, GATConfig, BOS_TOKEN_ID
torch.set_float32_matmul_precision('high')  # Enable TF32 for ~2x speedup
# For fast training, set 'BOS_TOKEN_ID' to 15 in 'sorl/gat_sim.py' 

gat_config = GATConfig(
    vocab_sizes=[BOS_TOKEN_ID+1, 6],  # 6 abstract tokens
    n_layer=12,
    n_head=6,
    n_embd=768,
    device="cuda" if torch.cuda.is_available() else "cpu",
    flex_kernel_options={
            "BLOCK_M": 32, "BLOCK_N": 32,
            "BLOCK_M1": 32, "BLOCK_N1": 64, "BLOCK_M2": 64, "BLOCK_N2": 32
        }
)
    
model = GAT(gat_config)
# model = model.to("cuda")
# model = torch.compile(model)

In [2]:
# ---- Copy & Paste Data Loader ----
from data.copy_paste import CopyPasteDataLoader

loader = CopyPasteDataLoader(vocab_size=16, max_token=10, seq_len=1, device='cpu')
tokens, loss_mask = loader.get_batch(2)

In [3]:
from sorl.neo_utils import sorl_search, compute_loss, sorl_evaluate

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)
batch_size = 16
memory_span = 1
attn_blocksize = 1792

for step in range(100): 
    optimizer.zero_grad()

    tokens, loss_mask = loader.get_batch(batch_size)
    # tokens = next(train_loader)

    # --- mixture of SoRL selection & deep supervision (avg. loss per iteration) ---
    with torch.no_grad(): 
        search_tokens, search_ppt, search_adv = sorl_search(tokens, model, n=2, K=4, max_iterations=1, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=1.0, loss_mask=loss_mask)
    
    # --- compute loss ---
    traj_loss, abs_loss = compute_loss(search_tokens, model, memory_span=memory_span, attn_blocksize=attn_blocksize, loss_mask=loss_mask)
    loss = traj_loss + abs_loss

    # GAPT
    
    # --- optimize --- 
    loss.backward() 
    optimizer.step()

    if step % 10 == 0: 
        with torch.no_grad(): 
            val_tokens, _, val_adv = sorl_evaluate(tokens, model, n=4, K=3, max_iterations=1, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=10.0, loss_mask=loss_mask)
            traj_loss, abs_loss = compute_loss(val_tokens, model, memory_span=memory_span, attn_blocksize=attn_blocksize, loss_mask=loss_mask)
        print(f"validation step {step} | traj_loss: {traj_loss.item():.2f} | abs_loss: {abs_loss.item():.2f} | search adv: {val_adv.item() * 100:.2f}%")

validation step 0 | traj_loss: 0.66 | abs_loss: 2.23 | search adv: 11.91%
validation step 10 | traj_loss: 0.49 | abs_loss: 5.79 | search adv: 2.46%
validation step 20 | traj_loss: 0.42 | abs_loss: 5.28 | search adv: 7.58%
validation step 30 | traj_loss: 0.35 | abs_loss: 7.49 | search adv: -7.48%
validation step 40 | traj_loss: 0.18 | abs_loss: 6.59 | search adv: -3.98%
validation step 50 | traj_loss: 0.12 | abs_loss: 7.15 | search adv: 13.30%
validation step 60 | traj_loss: 0.04 | abs_loss: 7.83 | search adv: 12.76%
validation step 70 | traj_loss: 0.01 | abs_loss: 8.43 | search adv: 11.41%
validation step 80 | traj_loss: 0.01 | abs_loss: 6.90 | search adv: 3.24%
validation step 90 | traj_loss: 0.00 | abs_loss: 7.83 | search adv: 5.37%


In [4]:
search_tokens

tensor([[15,  4, 17,  4, 15,  0, 17,  0, 15,  7, 17,  7, 15,  0, 17,  0, 15,  4,
         17,  4, 15,  4, 17,  4, 15,  3, 17,  3, 15,  6, 17,  6, 15,  3, 17,  3,
         15,  0, 17,  0, 15,  7, 17,  7, 15,  1, 17,  1, 15,  4, 18,  4, 15,  8,
         21,  8, 15,  1, 17,  1, 15,  2, 18,  2]])

In [5]:
# generate function implementation 
# ----------------------------------
# How to implement the 'generate' function? 
# - direct generation without search on prefix is easiest. 
# - I suspect 'search on prefix' requires 'max_iterations>1', otherwise model never learn to 'revise' from prior abstraction, only 'revise' from placeholder tokens
# - so, for now, we use 'max_iterations=0' to skip the prefix search

from sorl.gat_sim import generate

idx = tokens[:, :3].clone()
print(f"init   | idx: {idx[0].tolist()}")
for i in range(10): 
    idx = generate(model, idx, K=3, max_iterations=0, memory_span=1, attn_blocksize=1792, temperature=0.0)
    print(f"step {i+1} | idx: {idx[0].tolist()}")

init   | idx: [15, 4, 16]
step 1 | idx: [15, 4, 16, 4]
step 2 | idx: [15, 4, 16, 4, 3]
step 3 | idx: [15, 4, 16, 4, 3, 3]
step 4 | idx: [15, 4, 16, 4, 3, 3, 16]
step 5 | idx: [15, 4, 16, 4, 3, 3, 16, 3]
step 6 | idx: [15, 4, 16, 4, 3, 3, 16, 3, 3]
step 7 | idx: [15, 4, 16, 4, 3, 3, 16, 3, 3, 3]
step 8 | idx: [15, 4, 16, 4, 3, 3, 16, 3, 3, 3, 16]
step 9 | idx: [15, 4, 16, 4, 3, 3, 16, 3, 3, 3, 16, 3]
step 10 | idx: [15, 4, 16, 4, 3, 3, 16, 3, 3, 3, 16, 3, 3]


In [ ]:
# heuristic rollout
import os, glob, itertools
from pathlib import Path

# MPS specific data loader functional (single device ver.)
# --------------------------------------------
def _load_data_shard(file: Path):
    header = torch.from_file(str(file), False, 256, dtype=torch.int32) # header is 256 int32
    assert header[0] == 20240520, "magic number mismatch in the data .bin file"
    assert header[1] == 1, "unsupported version"
    num_tokens = int(header[2]) # number of tokens (claimed)
    with file.open("rb", buffering=0) as f:
        tokens = torch.empty(num_tokens, dtype=torch.uint16, pin_memory=False) # MPS requires pin_memory=False
        f.seek(256 * 4)
        nbytes = f.readinto(tokens.numpy()) # avoid bytes->array copy by @YouJiacheng
        assert nbytes == 2 * num_tokens, "number of tokens read does not match header"
    return tokens

def data_generator(filename_pattern: str, sequence_length: int, device: str): 

    filename_pattern = "data/fineweb10B/fineweb_train_*.bin"
    files = [Path(file) for file in sorted(glob.glob(filename_pattern))]
    file_iter = itertools.cycle(files)
    tokens, pos = _load_data_shard(next(file_iter)), 0
    while True: 
        # Concern 1. Doesn't this means end-of-file is never reached?
        if pos + sequence_length + 1 >= len(tokens): # not enough data left -> load a new file
            tokens, pos = _load_data_shard(next(file_iter)), 0

        idx = tokens[pos : pos + sequence_length + 1].unsqueeze(0).to(device=device, dtype=torch.int32, non_blocking=True)
        pos += sequence_length
        yield idx

# ------------------------------------------------

# Question 1. Should we separate inputs / targets? 
#             That's really asking whether we want to 'reflect' on inputs, or inputs + next token
#             from the generation perspective, we ought to reflect on inputs and predict next-tok

# Reflection 1. 
# - based on above thought, we ought to modify the 'forward' method to take 'inputs' & 'targets' separately
#   the ._forward_pass and recursion should be done only on 'inputs'


data = _load_data_shard(Path("data/fineweb10B/fineweb_train_000002.bin"))

train_loader = data_generator(filename_pattern="data/fineweb10B/fineweb_train_*.bin", sequence_length=256, device="cpu")
val_loader = data_generator(filename_pattern="data/fineweb10B/fineweb_val_000000.bin", sequence_length=256, device="cpu")

In [ ]:
# --- Benchmark Speed & Memory Cost --- 
from sorl.benchmark import run_benchmark_suite
import torch 


# Prepare data - TEST WITH SMALLER SEQUENCE FIRST
tokens = next(train_loader)

# Run benchmark
results = run_benchmark_suite(
    model, 
    tokens, 
    memory_span=1024, 
    num_runs=10
)